In [3]:
# pip install ligthfm

In [30]:
import numpy as np
import pandas as pd
from scipy.sparse import coo_matrix
from lightfm import LightFM
from lightfm.datasets import fetch_movielens
from lightfm.evaluation import precision_at_k
from lightfm.cross_validation import random_train_test_split

In [6]:
data = fetch_movielens(min_rating=4.0)

In [8]:
data.keys()

dict_keys(['train', 'test', 'item_features', 'item_feature_labels', 'item_labels'])

In [11]:
interactions = data['train'] + data['test']

In [12]:
interactions = interactions.tocoo()

In [17]:
df = pd.DataFrame({"user_id":interactions.row, "item_id":interactions.col, "rating":interactions.data})

In [18]:
df

,user_id,item_id,rating
0,0,0,5
1,0,2,4
2,0,5,5
3,0,6,4
4,0,8,5
...,...,...,...
55370,942,823,4
55371,942,839,4
55372,942,927,5
55373,942,942,5


In [19]:
df["item_label"] = df["item_id"].apply(lambda x : data["item_labels"][x])

In [21]:
df["rating"].value_counts()

rating
4    34174
5    21201
Name: count, dtype: int64

In [22]:
df["user_id"].nunique()

942

In [23]:
df["item_id"].nunique()

1447

In [24]:
df.groupby('item_label').size().sort_values(ascending=False).head(10)

item_label
S    6863
B    4344
M    4100
C    4014
T    3940
A    3445
R    3296
G    3177
F    3168
D    2644
dtype: int64

In [25]:
n_users = df["user_id"].max() + 1
n_items = df["item_id"].max() + 1

In [28]:
interactions_matrix = coo_matrix((df["rating"],(df["user_id"], df["item_id"])), shape=(n_users,n_items))

In [41]:
train, test = random_train_test_split(interactions_matrix, test_percentage=0.2, random_state=np.random.RandomState(42))

In [42]:
model = LightFM(loss="warp")

In [46]:
model.fit(train, epochs=40, num_threads=2)

In [47]:
precision = precision_at_k(model, test, k=5).mean()

In [48]:
precision

0.10271445

In [38]:
# 10.12%
# 0.05 - 0.15
# > 0.20

In [39]:
def recommend(model, n_items_total, item_labels, user_id, n_items=5):
    scores = model.predict(user_id, np.arange(n_items_total))
    top_items = np.argsort(-scores)[:n_items]
    return item_labels[top_items]

In [40]:
recommend(model, n_items, data["item_labels"], user_id=0)

array(['P', 'T', 'B', 'A', 'S'], dtype='<U1')